In [1]:
pip install -Uqq langchain.agents

ERROR: Could not find a version that satisfies the requirement langchain.agents (from versions: none)
ERROR: No matching distribution found for langchain.agents


In [2]:
pip install -Uqq groq

In [3]:
pip install -Uqq langchain-groq

In [4]:
pip install -Uqq deepagents

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [5]:
import os
from google.colab import userdata
from groq import Groq
from langchain.agents import create_agent
from langchain.tools import tool
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware
from deepagents.middleware.subagents import SubAgentMiddleware
from langchain.agents.middleware import TodoListMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [6]:
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

In [7]:
client = Groq()
for model in client.models.list().data:
  print(model.id)

openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
whisper-large-v3-turbo
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
allam-2-7b
whisper-large-v3


In [8]:
@tool
def search(query : str) -> str:
  """
  Search to find an answer about this query
  """
  return f"Search about {query}"

In [9]:
backend = StateBackend()

In [10]:
model = "groq:openai/gpt-oss-20b"
agent = create_agent(
    model = model,
    tools = [search],
    system_prompt = "you are a good researcher",
    checkpointer = InMemorySaver(),
    middleware=[
        FilesystemMiddleware(backend=backend),
        TodoListMiddleware(),
        SubAgentMiddleware(
            backend = backend,
            subagents = [
                {
                    "model" : model,
                    "name" : "researcher",
                    "description" : "you are a good researcher",
                    "system_prompt" : "Use tool to find the result of query",
                    "tools": [search],
                    "middleware":[]
                }
            ]
        )
    ]
)

In [11]:
config = {
    "configurable":{
        "thread_id":"s1"
        }
    }

In [14]:
while True:
  county = input("Enter the countey to know its capital? ")
  if county.strip().lower() == 'stop':
    break
  result = agent.invoke(
      {
          "messages":[{
              "role":"user",
              "content":f"what is the capital of {county}"
          }]
      },
      config=config
      )
  print(result["messages"][-1].content)

Enter the countey to know its capital? Egypt
The capital of Egypt is **Cairo**.
Enter the countey to know its capital? STOP 
